# OCR metrics and charts

Loads `outputs/pretrained_recheck.csv`, computes OCR metrics, and saves PNG charts plus a metrics CSV.

In [ ]:
from pathlib import Path
import math
from difflib import SequenceMatcher

import pandas as pd
import matplotlib.pyplot as plt
import jiwer

ROOT = Path.cwd()
if not (ROOT / 'outputs/pretrained_recheck.csv').exists():
    ROOT = Path('..').resolve()
INPUT = ROOT / 'outputs/pretrained_recheck.csv'
OUT = ROOT / 'outputs/ocr_charts'
OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(INPUT)
df.head()

In [ ]:
def aligned_counts(reference, prediction, unit):
    a = list(reference) if unit == 'char' else reference.split()
    b = list(prediction) if unit == 'char' else prediction.split()
    tp = fp = fn = distance = 0
    for tag, i1, i2, j1, j2 in SequenceMatcher(a=a, b=b).get_opcodes():
        left, right = i2 - i1, j2 - j1
        if tag == 'equal':
            tp += left
        elif tag == 'replace':
            matched = min(left, right)
            tp += matched
            fp += right - matched
            fn += left - matched
            distance += max(left, right)
        elif tag == 'insert':
            fp += right
            distance += right
        elif tag == 'delete':
            fn += left
            distance += left
    return tp, fp, fn, distance

def aggregate_metrics(frame, unit):
    counts = [aligned_counts(r, p, unit) for r, p in zip(frame.reference, frame.prediction)]
    tp = sum(x[0] for x in counts)
    fp = sum(x[1] for x in counts)
    fn = sum(x[2] for x in counts)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    rmse = math.sqrt(sum(x[3] ** 2 for x in counts) / len(counts))
    return precision, recall, f1, rmse

char_precision, char_recall, char_f1, char_rmse = aggregate_metrics(df, 'char')
word_precision, word_recall, word_f1, word_rmse = aggregate_metrics(df, 'word')
metrics = pd.DataFrame([
    ('exact_line_accuracy', (df.reference == df.prediction).mean()),
    ('cer', float(jiwer.cer(df.reference.tolist(), df.prediction.tolist()))),
    ('wer', float(jiwer.wer(df.reference.tolist(), df.prediction.tolist()))),
    ('character_precision', char_precision),
    ('character_recall', char_recall),
    ('character_f1', char_f1),
    ('word_precision', word_precision),
    ('word_recall', word_recall),
    ('word_f1', word_f1),
    ('character_edit_distance_rmse', char_rmse),
    ('word_edit_distance_rmse', word_rmse),
], columns=['metric', 'value'])
metrics.to_csv(OUT / 'pretrained_metrics.csv', index=False)
metrics

In [ ]:
plt.figure(figsize=(10, 5))
plot = metrics[~metrics.metric.str.contains('rmse')].copy()
plt.bar(plot.metric, plot.value)
plt.ylabel('Score')
plt.title('Pretrained TrOCR metrics')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(OUT / 'pretrained_metrics.png', dpi=180)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df.cer, bins=15, alpha=0.8, label='CER')
plt.hist(df.wer, bins=15, alpha=0.8, label='WER')
plt.xlabel('Error rate')
plt.ylabel('Number of images')
plt.title('Per-image OCR error distribution')
plt.legend()
plt.tight_layout()
plt.savefig(OUT / 'per_image_error_distribution.png', dpi=180)
plt.show()

In [ ]:
df.sort_values('cer', ascending=False).head(10)[['image_id', 'cer', 'wer', 'reference', 'prediction']].to_csv(OUT / 'worst_10_predictions.csv', index=False)
print(f'Saved charts and CSV files to: {OUT}')